# 6단계: QLoRA 파인튜닝 (조건 4) — 학습 (Colab)

목표: Colab T4(16GB)에서 Qwen2.5-Coder-7B-Instruct에 QLoRA 어댑터를 학습한다.
출력은 어댑터 하나이고, 예측 생성은 별도 노트북에서 vLLM으로 한다.

**조건 2·3과 다른 점 — AWQ는 학습할 수 없다.** 조건 2·3은 `Qwen2.5-Coder-7B-Instruct-AWQ`를 서빙했지만
AWQ는 추론 전용 양자화라 QLoRA를 얹을 수 없다. 그래서 학습은 **비양자화 원본 모델을 bitsandbytes NF4로**
불러와서 하고, 서빙은 조건 2·3과 동일한 AWQ 베이스에 어댑터만 얹는다(vLLM). 베이스 가중치가 조건 2·3과
같아야 "파인튜닝 효과"만 분리되기 때문이다. 학습(NF4)과 서빙(AWQ)의 양자화가 다른 건 알려진 절충이며
RESULTS.md에 기록한다.

**학습 설정** (로컬에서 잰 토큰 분포 기준)

| 항목 | 값 | 근거 |
|---|---|---|
| max_seq_len | 2048 | 초과 82개(1.2%)뿐. 중앙값 370, p90 856 |
| 초과 예제 | 학습에서 제외 | 잘리는 부분이 항상 정답 SQL이라 truncation은 불완전한 SQL을 가르침 |
| 정밀도 | fp16 | T4(Turing)는 bf16 미지원 |
| LoRA | r=64, alpha=16 | 브리프의 출발점 |
| 유효 배치 | 16 (1 × grad accum 16) | 16GB 한계 |

**사전 준비**: 런타임 유형 GPU(T4). `data/results/local_ft/train_messages.jsonl`(14MB) 업로드 필요.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets
import transformers, peft, trl, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| bnb", bitsandbytes.__version__)

## 학습 데이터 업로드

로컬에서 `uv run python scripts/build_finetune_dataset.py`로 만든 `train_messages.jsonl`(7,000개, 14MB)을 올린다.
프롬프트는 조건 2의 추론 프롬프트와 바이트 단위로 동일하다 — 어댑터가 서빙될 형식 그대로 학습된다.

In [ ]:
from pathlib import Path

DATA = Path("train_messages.jsonl")
if not DATA.exists():
    from google.colab import files

    files.upload()  # data/results/local_ft/train_messages.jsonl 선택
assert DATA.exists(), "train_messages.jsonl 업로드 필요"
print(DATA.stat().st_size // 1024, "KB")

In [ ]:
import json

from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"  # 비양자화 원본 (AWQ는 학습 불가)
MAX_SEQ_LEN = 2048

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

rows = [json.loads(line) for line in DATA.open(encoding="utf-8") if line.strip()]
texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows]

# Drop rather than truncate: what gets cut is always the tail, i.e. the gold SQL,
# so a truncated example would teach the model to emit incomplete queries.
kept = [t for t in texts if len(tokenizer(t)["input_ids"]) <= MAX_SEQ_LEN]
print(f"{len(kept)}/{len(texts)} kept, {len(texts) - len(kept)} dropped for exceeding {MAX_SEQ_LEN} tokens")

dataset = Dataset.from_dict({"text": kept})

## 4bit 로드 + LoRA 부착

NF4 이중 양자화로 7B를 T4에 올린다. gradient checkpointing을 켜야 활성값 메모리가 맞는다.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing: no bf16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.float16
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 학습

프롬프트가 아니라 **정답 SQL에만 손실을 건다**(completion-only). Qwen 채팅 템플릿의 어시스턴트 시작
토큰을 기준으로 마스킹한다 — 프롬프트까지 학습하면 스키마 텍스트를 외우는 데 용량을 쓰게 된다.

한 에폭은 약 437 스텝(6,918 / 16)이다. 중간에 세션이 끊겨도 되도록 50 스텝마다 체크포인트를 남긴다.

In [ ]:
from transformers import TrainingArguments
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer

collator = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n", tokenizer=tokenizer
)

args = TrainingArguments(
    output_dir="qwen_spider_qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    data_collator=collator,
    tokenizer=tokenizer,
)
trainer.train(resume_from_checkpoint=None)  # 재개할 때는 True

## 어댑터 저장 + 다운로드

어댑터만 저장하므로 수백 MB 수준이다. 받은 파일은 로컬 `data/results/local_ft/adapter/`에 넣는다.

In [ ]:
import shutil

ADAPTER_DIR = "qwen_spider_qlora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)

from google.colab import files

files.download(f"{ADAPTER_DIR}.zip")